# 00 · Setup check (Stage 0)

**Gate before any training** (spec §15 Stage 0). Verifies:
1. the uv environment imports (`torch`, `timm`, `openidh_model`),
2. pretrained weights load **fully offline** (`HF_HUB_OFFLINE=1`) from `weights/`,
3. the model builds and its parameter count is **exactly 44,224,396**,
4. a forward pass produces positive evidence of the right shape.

If offline construction fails here, it will fail on the (offline) supercomputer.
Run: `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/00_setup_check.ipynb`

In [1]:
import os, sys
from pathlib import Path

# Force offline: proves the HPC (no-network) path works (spec §1.4).
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Run from the repo root so `openidh_model` imports and configs/ paths resolve.
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
print("repo root:", _root)
print("HF_HUB_OFFLINE:", os.environ["HF_HUB_OFFLINE"])

repo root: /workspace
HF_HUB_OFFLINE: 1


In [2]:
import torch, timm
print("torch:", torch.__version__)
print("torchvision:", __import__("torchvision").__version__)
print("timm:", timm.__version__)
print("cuda available:", torch.cuda.is_available(), "(False here is fine; True on GB200)")

torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
timm: 1.0.28
cuda available: False (False here is fine; True on GB200)


/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Resolve paths (spec §1.2)

In [3]:
from openidh_model.utils.paths import load_paths
paths = load_paths("configs/paths.local.yaml")
paths.as_dict()

{'root': '/workspace',
 'data_dir': '/workspace/data/preprocessed/v1.0.0',
 'metadata_dir': '/workspace/data/metadata',
 'splits_dir': '/workspace/data/preprocessed/v1.0.0/_global/splits',
 'weights_dir': '/workspace/weights',
 'output_dir': '/workspace/runs'}

## Build the model offline and check the parameter count

In [4]:
from openidh_model.models.openidh import OpenIDH, IMG_MODALITIES

model = OpenIDH(weights_dir=paths.weights_dir)   # reads weights/*.safetensors, no network
breakdown = model.parameter_breakdown()
for k, v in breakdown.items():
    print(f"  {k:22s}: {v:,}")

EXPECTED = 44_224_396
assert breakdown["total"] == EXPECTED, f"param count {breakdown['total']:,} != {EXPECTED:,}"
print(f"\nOK — parameter count is exactly {EXPECTED:,}")

  trunk_single(3ch)     : 21,665,664
  trunk_unified(12ch)   : 22,550,400
  evidence_heads(x5)    : 3,850
  tabular_mlp           : 4,482
  total                 : 44,224,396

OK — parameter count is exactly 44,224,396


## Forward smoke test
Random tensors through all 6 classifiers → each emits `(B, 2)` strictly-positive evidence
`[e1, e0]`. (Volume scaling / fusion arrive in Stage A.)

In [5]:
model.eval()
B = 2
images  = {m: torch.randn(B, 3, 224, 224) for m in IMG_MODALITIES}
tabular = torch.randn(B, 2)   # [age_standardized, sex]
with torch.no_grad():
    ev = model.raw_evidence(images, tabular)

print("classifiers:", list(ev.keys()))
print("shapes     :", {k: tuple(v.shape) for k, v in ev.items()})
allpos = all((v > 0).all().item() for v in ev.values())
assert allpos and all(tuple(v.shape) == (B, 2) for v in ev.values())
print("OK — 6 classifiers, all evidence (B,2) and > 0")

classifiers: ['T1', 'T2', 'FLAIR', 'T1c', 'unified', 'tabular']
shapes     : {'T1': (2, 2), 'T2': (2, 2), 'FLAIR': (2, 2), 'T1c': (2, 2), 'unified': (2, 2), 'tabular': (2, 2)}
OK — 6 classifiers, all evidence (B,2) and > 0


## ✅ Stage 0 gate passed
Environment, offline weights, parameter count (44,224,396), and forward pass all verified.
Next: **Stage A** — dataloader (`01_data_check.ipynb`), losses + KL tests, fusion (the heart of the design).